In [1]:
import subprocess
import time
import requests
import os
from urllib.parse import urlparse, urlunparse, parse_qs, urlencode
import random
import subprocess
import time
import requests
import os
import subprocess
import random
import time
import psutil
import time
import random
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# --- Configuration ---
BAT_FILE_PATH = r"C:\Users\AdarshSingh\Downloads\start_edge_debug.bat"
DEBUGGING_URL = "http://127.0.0.1:9222/json/version"
SELENIUM_WAIT_TIMEOUT = 15

# --- Core Functions (No changes in this section) ---


def kill_edge_debug_instance():
    """
    Kill only the Edge instance launched with remote debugging (port 9222).
    This avoids killing your own Edge window or DevTools browser.
    """
    for proc in psutil.process_iter(attrs=['pid', 'name', 'cmdline']):
        try:
            if 'msedge.exe' in proc.info['name'].lower():
                cmdline = ' '.join(proc.info['cmdline']).lower()
                if '--remote-debugging-port=9222' in cmdline:
                    print(f"🧹 Killing debug Edge process (PID: {proc.info['pid']})...")
                    proc.kill()
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            continue

def launch_edge_with_bat():
    """Launches the Edge browser using the specified batch file."""
    print(f"Attempting to launch Edge via batch file: {BAT_FILE_PATH}")
    try:
        subprocess.Popen([BAT_FILE_PATH], shell=True, creationflags=subprocess.CREATE_NEW_CONSOLE)
        return True
    except FileNotFoundError:
        print(f"ERROR: Batch file not found at '{BAT_FILE_PATH}'. Please check the path.")
        return False
    except Exception as e:
        print(f"Failed to run batch file: {e}")
        return False

def wait_for_devtools_port(timeout=15):
    """Waits for the Edge DevTools debugging port to become active."""
    start_time = time.time()
    print(f"Waiting for Edge DevTools on {DEBUGGING_URL} (timeout: {timeout}s)...")
    while time.time() - start_time < timeout:
        try:
            res = requests.get(DEBUGGING_URL, timeout=1)
            if res.status_code == 200:
                print("Edge DevTools is active!")
                return True
        except requests.exceptions.RequestException:
            pass
        time.sleep(0.5)
    print("Timeout: DevTools port not available.")
    return False

def connect_selenium():
    """Connects Selenium to the already running Edge debug instance."""
    print("Connecting Selenium to Edge debugger...")
    edge_options = Options()
    edge_options.debugger_address = "127.0.0.1:9222"
    try:
        driver = webdriver.Edge(options=edge_options)
        print("✅ Selenium successfully connected to Edge.")
        return driver
    except Exception as e:
        print(f"❌ Failed to connect Selenium to Edge: {e}")
        return None

def perform_linkedin_search(driver, query):
    """Finds the search box and performs a search on LinkedIn."""
    print(f"\nStep 3: Performing LinkedIn search for: '{query}'")
    try:
        search_box_selector = "input.search-global-typeahead__input"
        search_box = WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, search_box_selector))
        )
        search_box.clear()
        search_box.send_keys(query)
        search_box.send_keys(Keys.ENTER)
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT).until(
            EC.url_contains("/search/results/")
        )
        print("✅ Search successful, results page is loading.")
        time.sleep(2)
        return True
    except Exception as e:
        print(f"❌ Error during LinkedIn search: {e}")
        return False

def navigate_to_people_filter(driver):
    """Navigates to the 'People' filtered results page."""
    print("\nStep 4: Navigating to 'People' results...")
    try:
        people_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[normalize-space()='People']"))
        )
        people_button.click()
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT).until(EC.url_contains("/people/"))
        print("✅ Successfully filtered for 'People' by clicking button.")
        time.sleep(3)
        return True
    except TimeoutException:
        print("   -> Could not click 'People' button, using URL modification fallback...")
        current_url = driver.current_url
        if "/search/results/all/" in current_url:
            people_url = current_url.replace("/search/results/all/", "/search/results/people/")
            driver.get(people_url)
            WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT).until(EC.url_contains("/people/"))
            time.sleep(3)
            print("✅ Successfully filtered for 'People' by modifying URL.")
            return True
        elif "/people/" in driver.current_url:
            print("✅ Already on a people results page.")
            return True
        else:
            print("❌ Cannot navigate to people results. Current URL is not a known search results page.")
            return False

# --- THIS FUNCTION HAS BEEN MODIFIED FOR DYNAMIC PAGINATION ---
def scrape_search_results(driver, max_pages_limit=20):
    """
    Iterates through search pages and scrapes profile links until a page yields no new
    profiles, or the max_pages_limit is reached.
    """
    print("\nStep 5: Beginning DYNAMIC page iteration and profile scraping...")
    
    all_profile_links = set()
    
    initial_url = driver.current_url
    print(f"   -> Template URL for pagination: {initial_url}")

    parsed_url = urlparse(initial_url)
    base_query_params = parse_qs(parsed_url.query)
    
    # max_pages_limit acts as a safety stop, but the loop will break earlier if results end.
    for page_number in range(1, max_pages_limit + 1):
        base_query_params['page'] = [str(page_number)]
        
        new_query_string = urlencode(base_query_params, doseq=True)
        paginated_url = urlunparse((
            parsed_url.scheme,
            parsed_url.netloc,
            parsed_url.path,
            parsed_url.params,
            new_query_string,
            parsed_url.fragment
        ))

        print(f"\n--- Processing Page {page_number} ---")
        print(f"   -> Navigating to: {paginated_url}")
        driver.get(paginated_url)
        time.sleep(random.uniform(2.5, 4.5))

        try:
            main_container_selector = "div.search-marvel-srp"
            main_container = WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, main_container_selector))
            )
            
            profile_link_elements = main_container.find_elements(By.XPATH, ".//a[contains(@href, 'linkedin.com/in/')]")
            
            # If the page has no profile links at all, it's the end.
            if not profile_link_elements:
                print("   -> No profile link elements found on this page. This is the end of the results.")
                break
                
            found_this_page = 0
            for link_element in profile_link_elements:
                href = link_element.get_attribute('href')
                if href:
                    parsed_href = urlparse(href)
                    clean_url = urlunparse((parsed_href.scheme, parsed_href.netloc, parsed_href.path, '', '', ''))
                    
                    if href not in all_profile_links:
                        all_profile_links.add(clean_url)
                        found_this_page += 1
            
            print(f"   -> ✅ Found {found_this_page} new unique profile links on this page.")

            # --- DYNAMIC TERMINATION CHECK ---
            # If we processed a page but didn't add any new links to our set,
            # it means we've reached the end of the unique results.
            if found_this_page == 0:
                print("   -> No *new* unique profiles were found on this page. Ending pagination.")
                break

        except TimeoutException:
            print(f"   -> ❌ Could not find the main search container on page {page_number}. Assuming end of results.")
            break # Stop if the page structure is missing.
        except Exception as e:
            print(f"   -> ❌ An error occurred during scraping on page {page_number}: {e}")
            break # Stop on other critical errors.
    
    print(f"\n--- Finished scraping after processing {page_number} pages. ---")
    return list(all_profile_links)

final_links=[]
if __name__ == "__main__":
    print("--- LinkedIn Scraper: Final Dynamic Version ---")

    print("\nStep 1: Initializing Browser...")
    if not wait_for_devtools_port(timeout=5):
        print("   -> DevTools port not ready. Trying to launch via batch file...")
        if not launch_edge_with_bat() or not wait_for_devtools_port():
              print("❌ Could not start or connect to Edge. Please start it manually and rerun. Exiting.")
              exit(1)

    driver = connect_selenium()
    if driver:
        try:
            print("\nStep 2: Preparing LinkedIn...")
            if "linkedin.com/feed/" not in driver.current_url:
                driver.get("https://www.linkedin.com/feed/")
            
            print("   -> IMPORTANT: Please ensure you are logged in.")
            print("   -> Waiting 10 seconds for you to check...")
            time.sleep(10)

            user_query = input("Enter LinkedIn search keywords and press Enter: ")
            
            if perform_linkedin_search(driver, user_query):
                if navigate_to_people_filter(driver):
                    # The function will now stop automatically when results end.
                    # The `max_pages_limit` is just a safety cap.
                    final_links+= scrape_search_results(driver, max_pages_limit=20)
                    
                    if final_links:
                        print(f"\n\n--- ✅ SUCCESS: Collected {len(final_links)} Unique Profile Links ---")
                        for idx, link in enumerate(final_links):
                            print(f"{idx + 1}. {link}")
                    else:
                        print("\n\n--- No profile links were collected. ---")

            print("\n✅ Script finished its current flow.")

        except Exception as main_e:
            print(f"\n--- An critical error occurred in the main script ---")
            import traceback
            traceback.print_exc()
        finally:
            kill_edge_debug_instance()
            delay = random.uniform(1, 2)
            print(f"⏳ Waiting {delay:.2f} seconds before next iteration...")
            time.sleep(delay)

    else:
        print("❌ Failed to initialize Selenium WebDriver. Exiting.")

--- LinkedIn Scraper: Final Dynamic Version ---

Step 1: Initializing Browser...
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 5s)...
Timeout: DevTools port not available.
   -> DevTools port not ready. Trying to launch via batch file...
Attempting to launch Edge via batch file: C:\Users\AdarshSingh\Downloads\start_edge_debug.bat
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 15s)...
Edge DevTools is active!
Connecting Selenium to Edge debugger...
✅ Selenium successfully connected to Edge.

Step 2: Preparing LinkedIn...
   -> IMPORTANT: Please ensure you are logged in.
   -> Waiting 10 seconds for you to check...


Enter LinkedIn search keywords and press Enter:  Gartner



Step 3: Performing LinkedIn search for: 'Gartner'
✅ Search successful, results page is loading.

Step 4: Navigating to 'People' results...
   -> Could not click 'People' button, using URL modification fallback...
✅ Successfully filtered for 'People' by modifying URL.

Step 5: Beginning DYNAMIC page iteration and profile scraping...
   -> Template URL for pagination: https://www.linkedin.com/search/results/people/?keywords=Gartner&origin=GLOBAL_SEARCH_HEADER&sid=JTP

--- Processing Page 1 ---
   -> Navigating to: https://www.linkedin.com/search/results/people/?keywords=Gartner&origin=GLOBAL_SEARCH_HEADER&sid=JTP&page=1
   -> ✅ Found 39 new unique profile links on this page.

--- Processing Page 2 ---
   -> Navigating to: https://www.linkedin.com/search/results/people/?keywords=Gartner&origin=GLOBAL_SEARCH_HEADER&sid=JTP&page=2
   -> ✅ Found 33 new unique profile links on this page.

--- Processing Page 3 ---
   -> Navigating to: https://www.linkedin.com/search/results/people/?keywords=

In [2]:


# --- Configuration ---
BAT_FILE_PATH = r"C:\Users\AdarshSingh\Downloads\start_edge_debug.bat"
DEBUGGING_URL = "http://127.0.0.1:9222/json/version"
SELENIUM_WAIT_TIMEOUT = 10

# --- Helper Functions (Unchanged) ---

def launch_edge_with_bat():
    print(f"Attempting to launch Edge via batch file: {BAT_FILE_PATH}")
    try:
        subprocess.Popen([BAT_FILE_PATH], shell=True, creationflags=subprocess.CREATE_NEW_CONSOLE)
        return True
    except Exception as e:
        print(f"ERROR: Failed to run batch file: {e}")
        return False

def wait_for_devtools_port(timeout=15):
    start_time = time.time()
    print(f"Waiting for Edge DevTools on {DEBUGGING_URL} (timeout: {timeout}s)...")
    while time.time() - start_time < timeout:
        try:
            res = requests.get(DEBUGGING_URL, timeout=1)
            if res.status_code == 200:
                print("Edge DevTools is active!")
                return True
        except requests.exceptions.RequestException:
            pass
        time.sleep(0.5)
    print("Timeout: DevTools port not available.")
    return False

def connect_selenium():
    print("Connecting Selenium to Edge debugger...")
    edge_options = Options()
    edge_options.debugger_address = "127.0.0.1:9222"
    try:
        driver = webdriver.Edge(options=edge_options)
        print("✅ Selenium successfully connected to Edge.")
        return driver
    except Exception as e:
        print(f"❌ Failed to connect Selenium to Edge: {e}")
        return None

# def pause_for_debug(message):
#     print(f"\n--- DEBUG PAUSE ---")
#     print(f"   ACTION: {message}")
#     input("   -> Press Enter to continue...")

# --- Core Logic Functions ---

def get_profile_name(driver):
    """Scrapes the full name from the profile page."""
    print("\nStep 3: Scraping profile name...")
    try:
        name_element_xpath = "//h1"
        name_element = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, name_element_xpath))
        )
        full_name = name_element.text
        return full_name
    except Exception as e:
        print(f"   -> ❌ Could not scrape profile name: {e}")
        return None

# --- UPDATED FUNCTION WITH ROBUST DROPDOWN CLICK ---
# def send_connection_request(driver, full_name):
#     """
#     Checks for a 'More' button first, then uses the full scraped name to find
#     the precise element to click using a robust JavaScript click.
#     """
#     print("\nStep 4: Attempting to click the 'Connect' button...")
#     main_card_xpath = "//div[contains(@class, 'ph5') and contains(@class, 'pb5')]"
#     more_button_xpath = f"{main_card_xpath}//button[@aria-label='More actions']"

#     try:
#         direct_connect_button_xpath = f"{main_card_xpath}//button[@aria-label='Invite {full_name} to connect']"
#         pause_for_debug(f"Searching for direct 'Connect' button with EXACT selector:\n   {direct_connect_button_xpath}")
        
#         connect_button = WebDriverWait(driver, 5).until(
#             EC.presence_of_element_located((By.XPATH, direct_connect_button_xpath))
#         )
#         driver.execute_script("arguments[0].click();", connect_button)
#         print("   -> ✅ Success! Direct 'Connect' button was clicked.")
#         return True
        
#     except NoSuchElementException:
#         print("   -> 'Direct' button not found. Looking for a More 'Connect' button...")
#         try:
#             more_button = driver.find_element(By.XPATH, more_button_xpath)
#             print("   -> Found 'More' button. Executing dropdown workflow.")
#             pause_for_debug("Will now click the 'More' button.")
#             driver.execute_script("arguments[0].click();", more_button)
#             time.sleep(1)
    
#             dropdown_connect_div_xpath = f"//div[@role='button' and @aria-label='Invite {full_name} to connect']"
#             pause_for_debug(f"Searching for dropdown 'Connect' option with EXACT selector:\n   {dropdown_connect_div_xpath}")
            
#             # THIS IS THE CORRECTED PART: Find the element, then use JS to click it.
#             connect_option = WebDriverWait(driver, 5).until(
#                 EC.presence_of_element_located((By.XPATH, dropdown_connect_div_xpath))
#             )
#             driver.execute_script("arguments[0].click();", connect_option)
            
#             print("   -> ✅ Success! Clicked 'Connect' option from the 'More' menu.")
#             return True
#         except Exception as e:
#             print(f"   -> ❌ Failed to find direct 'Connect' button as well. Error: {e}")
#             return False

# --- REPLACE YOUR OLD FUNCTION WITH THIS DEFINITIVE VERSION ---
def send_connection_request(driver, full_name):
    invite_button_xpath =  f"//div[@role='button' and @aria-label='Invite {full_name} to connect']"
    try:
        invite_button = driver.find_element(By.XPATH, invite_button_xpath)
        print(f"   -> ✅ Found invite button by aria-label. Clicking...")
        driver.execute_script("arguments[0].click();", invite_button)
        return True
    except NoSuchElementException:
        try:
            direct_connect_button_xpath = f"//button[@aria-label='Invite {full_name} to connect']"
            connect_button = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, direct_connect_button_xpath))
            )
            driver.execute_script("arguments[0].click();", connect_button)
            print("   -> ✅ Success! Direct 'Connect' button was clicked.")
            return True
            
        except NoSuchElementException:
            return False


def handle_send_invitation_modal(driver):
    """Handles the final step of sending the invitation."""
    print("\nStep 5: Handling the 'Send Invitation' modal...")
    try:
        send_button_xpath = "//button[@aria-label='Send without a note']"
        
        send_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, send_button_xpath))
        )
        
        send_button.click() 
        print("   -> ✅ Invitation sent successfully!")
        return True
            
    except TimeoutException:
        print("\n   -> 🟡 Warning: Could not find the 'Send without a note' modal/button.")
        print("      (This may be OKAY if the invitation was sent instantly).")
        return False

In [3]:


def kill_edge_debug_instance():
    """
    Kill only the Edge instance launched with remote debugging (port 9222).
    This avoids killing your own Edge window or DevTools browser.
    """
    for proc in psutil.process_iter(attrs=['pid', 'name', 'cmdline']):
        try:
            if 'msedge.exe' in proc.info['name'].lower():
                cmdline = ' '.join(proc.info['cmdline']).lower()
                if '--remote-debugging-port=9222' in cmdline:
                    print(f"🧹 Killing debug Edge process (PID: {proc.info['pid']})...")
                    proc.kill()
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            continue

import random
sampled_links = random.sample(final_links, min(50, len(final_links)))
final_links = sampled_links



# --- Main Execution ---
if __name__ == "__main__":
   
    for TEST_PROFILE_URL in final_links:
        print(f"\n🔁 Starting processing for: {TEST_PROFILE_URL}")

        # STEP 0: Kill old Edge instance before starting a fresh one
        kill_edge_debug_instance()
        time.sleep(1)  # Give OS a second to release the process

        print("\nStep 1: Initializing Browser...")
        if not wait_for_devtools_port(timeout=5):
            print("   -> DevTools port not ready. Trying to launch via batch file...")
            if not launch_edge_with_bat() or not wait_for_devtools_port():
                print("❌ Could not start or connect to Edge. Please start it manually and rerun. Skipping profile.")
                continue

        driver = connect_selenium()
        if driver:
            try:
                print(f"\nStep 2: Navigating to profile -> {TEST_PROFILE_URL}")
                driver.get(TEST_PROFILE_URL)
                time.sleep(2)  # Let the page render

                full_name = get_profile_name(driver)
                if full_name:
                    if send_connection_request(driver, full_name):
                        handle_send_invitation_modal(driver)
                    else:
                        print("   -> ℹ️ Could not send connection (already connected or button missing).")
                else:
                    print("   -> ❌ Could not extract full name. Skipping profile.")

                print("\n✅ Profile processing complete.")

            except Exception as main_e:
                print(f"\n--- A critical error occurred while processing {TEST_PROFILE_URL} ---")
                import traceback
                traceback.print_exc()
            finally:
                if driver:
                    print("🧹 Closing browser...")
                    try:
                        driver.quit()
                    except Exception as quit_err:
                        print(f"   -> Warning: Error during browser quit: {quit_err}")

                # STEP N: Ensure browser is killed after driver.quit() (safety net)
                kill_edge_debug_instance()

                delay = random.uniform(1, 2)
                print(f"⏳ Waiting {delay:.2f} seconds before next iteration...")
                time.sleep(delay)

        else:
            print("❌ Failed to initialize Selenium WebDriver. Skipping profile.")



🔁 Starting processing for: https://www.linkedin.com/in/monica-bhatia-8524b9a2

Step 1: Initializing Browser...
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 5s)...
Timeout: DevTools port not available.
   -> DevTools port not ready. Trying to launch via batch file...
Attempting to launch Edge via batch file: C:\Users\AdarshSingh\Downloads\start_edge_debug.bat
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 15s)...
Edge DevTools is active!
Connecting Selenium to Edge debugger...
✅ Selenium successfully connected to Edge.

Step 2: Navigating to profile -> https://www.linkedin.com/in/monica-bhatia-8524b9a2

Step 3: Scraping profile name...
   -> ✅ Success! Direct 'Connect' button was clicked.

Step 5: Handling the 'Send Invitation' modal...
   -> ✅ Invitation sent successfully!

✅ Profile processing complete.
🧹 Closing browser...
🧹 Killing debug Edge process (PID: 5820)...
🧹 Killing debug Edge process (PID: 9356)...
🧹 Killing debug Ed

Traceback (most recent call last):
  File "C:\Users\AdarshSingh\AppData\Local\Temp\ipykernel_5236\2196641357.py", line 114, in send_connection_request
    invite_button = driver.find_element(By.XPATH, invite_button_xpath)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 914, in find_element
    return self.execute(Command.FIND_ELEMENT, {"using": by, "value": value})["value"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 447, in execute
    self.error_handler.check_response(response)
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.NoSuchElementException: Message: no such e

🧹 Killing debug Edge process (PID: 7004)...
🧹 Killing debug Edge process (PID: 12888)...
🧹 Killing debug Edge process (PID: 18536)...
🧹 Killing debug Edge process (PID: 20172)...
🧹 Killing debug Edge process (PID: 20268)...
🧹 Killing debug Edge process (PID: 20736)...
⏳ Waiting 1.47 seconds before next iteration...

🔁 Starting processing for: https://www.linkedin.com/in/ACoAADQn2q8BtslgeZR9_kUl_YMeUbWrJzoCRxc

Step 1: Initializing Browser...
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 5s)...
Timeout: DevTools port not available.
   -> DevTools port not ready. Trying to launch via batch file...
Attempting to launch Edge via batch file: C:\Users\AdarshSingh\Downloads\start_edge_debug.bat
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 15s)...
Edge DevTools is active!
Connecting Selenium to Edge debugger...
✅ Selenium successfully connected to Edge.

Step 2: Navigating to profile -> https://www.linkedin.com/in/ACoAADQn2q8BtslgeZR9_kU

Traceback (most recent call last):
  File "C:\Users\AdarshSingh\AppData\Local\Temp\ipykernel_5236\2196641357.py", line 114, in send_connection_request
    invite_button = driver.find_element(By.XPATH, invite_button_xpath)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 914, in find_element
    return self.execute(Command.FIND_ELEMENT, {"using": by, "value": value})["value"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 447, in execute
    self.error_handler.check_response(response)
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.NoSuchElementException: Message: no such e

🧹 Killing debug Edge process (PID: 3400)...
⏳ Waiting 1.09 seconds before next iteration...

🔁 Starting processing for: https://www.linkedin.com/in/ACoAACegI5UBANd5yNuqNpKm81-917D5Slim1sY

Step 1: Initializing Browser...
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 5s)...
Timeout: DevTools port not available.
   -> DevTools port not ready. Trying to launch via batch file...
Attempting to launch Edge via batch file: C:\Users\AdarshSingh\Downloads\start_edge_debug.bat
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 15s)...
Edge DevTools is active!
Connecting Selenium to Edge debugger...
✅ Selenium successfully connected to Edge.

Step 2: Navigating to profile -> https://www.linkedin.com/in/ACoAACegI5UBANd5yNuqNpKm81-917D5Slim1sY

Step 3: Scraping profile name...
   -> ✅ Success! Direct 'Connect' button was clicked.

Step 5: Handling the 'Send Invitation' modal...
   -> ✅ Invitation sent successfully!

✅ Profile processing complete.
🧹

...

🧹 Killing debug Edge process (PID: 752)...
🧹 Killing debug Edge process (PID: 3260)...
🧹 Killing debug Edge process (PID: 14396)...
🧹 Killing debug Edge process (PID: 16228)...
🧹 Killing debug Edge process (PID: 18816)...
🧹 Killing debug Edge process (PID: 19256)...
⏳ Waiting 1.60 seconds before next iteration...

🔁 Starting processing for: https://www.linkedin.com/in/nandini-pandey-952811122

Step 1: Initializing Browser...
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 5s)...
Timeout: DevTools port not available.
   -> DevTools port not ready. Trying to launch via batch file...
Attempting to launch Edge via batch file: C:\Users\AdarshSingh\Downloads\start_edge_debug.bat
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 15s)...
Edge DevTools is active!
Connecting Selenium to Edge debugger...
✅ Selenium successfully connected to Edge.

Step 2: Navigating to profile -> https://www.linkedin.com/in/nandini-pandey-952811122

Step 3: Scrapi

Traceback (most recent call last):
  File "C:\Users\AdarshSingh\AppData\Local\Temp\ipykernel_5236\2196641357.py", line 114, in send_connection_request
    invite_button = driver.find_element(By.XPATH, invite_button_xpath)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 914, in find_element
    return self.execute(Command.FIND_ELEMENT, {"using": by, "value": value})["value"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 447, in execute
    self.error_handler.check_response(response)
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.NoSuchElementException: Message: no such e

🧹 Killing debug Edge process (PID: 5936)...
🧹 Killing debug Edge process (PID: 7272)...
⏳ Waiting 1.94 seconds before next iteration...

🔁 Starting processing for: https://www.linkedin.com/in/neetika-narang-416a1235

Step 1: Initializing Browser...
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 5s)...
Timeout: DevTools port not available.
   -> DevTools port not ready. Trying to launch via batch file...
Attempting to launch Edge via batch file: C:\Users\AdarshSingh\Downloads\start_edge_debug.bat
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 15s)...
Edge DevTools is active!
Connecting Selenium to Edge debugger...
✅ Selenium successfully connected to Edge.

Step 2: Navigating to profile -> https://www.linkedin.com/in/neetika-narang-416a1235

Step 3: Scraping profile name...
   -> ✅ Success! Direct 'Connect' button was clicked.

Step 5: Handling the 'Send Invitation' modal...
   -> ✅ Invitation sent successfully!

✅ Profile processing

Traceback (most recent call last):
  File "C:\Users\AdarshSingh\AppData\Local\Temp\ipykernel_5236\2196641357.py", line 114, in send_connection_request
    invite_button = driver.find_element(By.XPATH, invite_button_xpath)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 914, in find_element
    return self.execute(Command.FIND_ELEMENT, {"using": by, "value": value})["value"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 447, in execute
    self.error_handler.check_response(response)
  File "C:\Users\AdarshSingh\anaconda3\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 232, in check_response
    raise exception_class(message, screen, stacktrace)
selenium.common.exceptions.NoSuchElementException: Message: no such e

🧹 Killing debug Edge process (PID: 13696)...
🧹 Killing debug Edge process (PID: 21336)...
🧹 Killing debug Edge process (PID: 22132)...
🧹 Killing debug Edge process (PID: 24704)...
🧹 Killing debug Edge process (PID: 24952)...
🧹 Killing debug Edge process (PID: 25328)...
🧹 Killing debug Edge process (PID: 25828)...
🧹 Killing debug Edge process (PID: 25960)...
⏳ Waiting 1.17 seconds before next iteration...

🔁 Starting processing for: https://www.linkedin.com/in/mayank-verma-779400104

Step 1: Initializing Browser...
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 5s)...
Timeout: DevTools port not available.
   -> DevTools port not ready. Trying to launch via batch file...
Attempting to launch Edge via batch file: C:\Users\AdarshSingh\Downloads\start_edge_debug.bat
Waiting for Edge DevTools on http://127.0.0.1:9222/json/version (timeout: 15s)...
Edge DevTools is active!
Connecting Selenium to Edge debugger...
✅ Selenium successfully connected to Edge.

Step 2: Na